# Learning Classical Density Functionals for Ionic Fluids
### Python/PyTorch Tutorial

This notebook follows the approach of:

> **Bui & Cox, Phys. Rev. Lett. 134, 148001 (2025)**  
> *Learning Classical Density Functionals for Ionic Fluids*

which extends the Sammüller neural functional theory (PNAS 2023) to **ionic fluids** by combining:
- Grand-canonical MC for a short-range "mimic" system
- Local Molecular Field Theory (LMFT) to handle long-range Coulomb interactions
- Two-channel neural networks that learn c₁(z; [ρ₊, ρ₋])

See also `tutorial_neural_dft.ipynb` for the simpler 1D hard-rod case.

**Contents:**
1. The Restricted Primitive Model (RPM)
2. LMFT and the Mimic System
3. GCMC Data Generation
4. Two-Channel Neural Functional
5. DFT Self-Consistency and Predictions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

## Part 1: The Restricted Primitive Model (RPM)

The RPM is the simplest model for an ionic fluid: equal-diameter (σ) hard spheres carrying charges ±q embedded in a uniform dielectric continuum with permittivity ε.

The pair potential between ions i and j is:
$$u_{ij}(r) = \begin{cases} \infty & r < \sigma \\ \frac{q_i q_j}{\varepsilon r} & r \geq \sigma \end{cases}$$

**Challenge for neural functionals:** The Coulomb potential is long-ranged (1/r), so the direct correlation function c₁(z; [ρ]) is **nonlocal** — it depends on density profiles far from z. This breaks the locality assumption of the Sammüller approach.

In [ ]:
# Illustrate the RPM pair potential vs the mimic short-range potential
r = np.linspace(1.01, 6.0, 500)  # r in units of sigma

kappa_inv = 1.8  # κ⁻¹ in σ units (Bui & Cox default)
kappa = 1.0 / kappa_inv

# Coulomb: 1/r (in units where q²/ε = 1)
u_full = 1.0 / r

# Short-range mimic: v₀(r) = erfc(κr)/r
v0 = np.vectorize(lambda x: __import__('math').erfc(kappa * x) / x)(r)

# Long-range mean-field part: v₁(r) = erf(κr)/r = 1/r - erfc(κr)/r
v1 = u_full - v0

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r, u_full, 'k-',  lw=2, label=r'$1/r$ (full Coulomb)')
ax.plot(r, v0,    'b-',  lw=2, label=r'$v_0(r) = \mathrm{erfc}(\kappa r)/r$ (mimic SR)')
ax.plot(r, v1,    'r--', lw=2, label=r'$v_1(r) = \mathrm{erf}(\kappa r)/r$ (mean-field LR)')
ax.axvline(kappa_inv, color='gray', ls=':', label=fr'$\kappa^{{-1}} = {kappa_inv}\sigma$')
ax.set_xlabel(r'$r/\sigma$')
ax.set_ylabel('Pair potential (a.u.)')
ax.set_title('LMFT splitting of Coulomb potential')
ax.legend()
ax.set_ylim(-0.1, 2.0)
ax.set_xlim(1.0, 6.0)
plt.tight_layout()
plt.show()
print(f'At r=σ: v0={np.vectorize(lambda x: __import__("math").erfc(kappa*x)/x)(np.array([1.0]))[0]:.3f}, '
      f'v1={1 - np.vectorize(lambda x: __import__("math").erfc(kappa*x)/x)(np.array([1.0]))[0]:.3f}')

## Part 2: LMFT and the Mimic System

**Local Molecular Field Theory** (Weeks et al.) provides a systematic way to handle long-range interactions:

$$\frac{1}{r} = \underbrace{\frac{\mathrm{erfc}(\kappa r)}{r}}_{v_0(r),\ \text{short-range}} + \underbrace{\frac{\mathrm{erf}(\kappa r)}{r}}_{v_1(r),\ \text{long-range, mean-field}}$$

The **mimic system** uses only $v_0$ for pair interactions, plus an external electrostatic potential $\phi_R(z)$ that is chosen so that the mimic system has the **same one-body density** as the full system:
$$\rho_{R,\nu}(\mathbf{r}) = \rho_\nu(\mathbf{r})$$

Key advantage: each particle in the mimic system is electroneutral (point charge + compensating Gaussian), enabling straightforward GCMC with insertion/deletion/swapping moves.

The one-body direct correlation function for the mimic system (Bui & Cox eq. 3):
$$c^{(1)}_{R,\nu}(z) = \ln\left(\Lambda^3 \rho_{R,\nu}(z)\right) + \beta V_{R,\nu}(z) + \beta q_\nu \phi_R(z) - \beta\mu_{R,\nu}$$

With $\Lambda = 1$: $c^{(1)}_{R,\nu}(z) = \ln\rho_{R,\nu}(z) + \beta V_{R,\nu}(z) + \beta q_\nu \phi_R(z) - \beta\mu_{R,\nu}$

In [ ]:
# Show the LMFT mean-field correction function
from mlip_mc.dft.rpm_sim import lmft_potential

Lz = 20.0
n_bins_z = 200
dz = Lz / n_bins_z
z = np.linspace(dz/2, Lz - dz/2, n_bins_z)

# Example charge density: symmetric RPM near a wall (mock)
rho_charge = np.zeros(n_bins_z)
# Positive layer near z=2, negative layer near z=4 (typical EDL)
rho_charge += 0.5 * np.exp(-((z - 2.0)**2) / 0.5)
rho_charge -= 0.5 * np.exp(-((z - 4.0)**2) / 0.5)

phi_mf = lmft_potential(rho_charge, z, kappa=kappa, Lz=Lz)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax1.plot(z, rho_charge, 'k-')
ax1.set_ylabel(r'$\rho_+(z) - \rho_-(z)$')
ax1.set_title('Charge density and LMFT mean-field potential')
ax2.plot(z, phi_mf, 'r-')
ax2.set_ylabel(r'$\phi_{\rm MF}(z)$ [kT]')
ax2.set_xlabel(r'$z/\sigma$')
plt.tight_layout()
plt.show()

## Part 3: GCMC Data Generation

We generate training data by running GCMC simulations of the mimic RPM system under random combinations of:
- External potentials $V_{R,\pm}(z)$ (sinusoids, ramps)
- External electrostatic potential $\phi_R(z)$ (sinusoidal)
- Chemical potentials $\beta\mu_{R,\pm}$

From each simulation we extract $\{\rho_+(z), \rho_-(z), c^{(1)}_{R,+}(z), c^{(1)}_{R,-}(z)\}$.

**Note:** The full paper uses ~2500 simulations. For this tutorial we use a small set, or mock analytical data.

In [ ]:
# Flag: use mock data (fast) or run actual GCMC (slow)
USE_MOCK_DATA = True  # Set to False to run actual GCMC simulations

N_SAMPLES = 50  # number of training simulations (use 500+ for real training)

In [ ]:
from mlip_mc.dft.rpm_sim import simulate_rpm

def generate_random_vext_ionic(L, rng):
    """Random external potential for one species: sum of sinusoids."""
    n_modes = rng.integers(1, 4)
    amplitudes = rng.uniform(-1.0, 1.0, n_modes)
    phases = rng.uniform(0, 2*np.pi, n_modes)
    def vext_fn(z):
        return sum(A * np.sin(2*np.pi*k*(z/L) + phi)
                   for k, (A, phi) in enumerate(zip(amplitudes, phases), 1))
    return vext_fn

def make_mock_c1(rho_p, rho_m, mu_p, mu_m, phi_vals, beta=1.0):
    """Analytical approximation for c1 in RPM (ideal + hard-sphere correction)."""
    eta = rho_p + rho_m  # total packing fraction (sigma=1)
    c1_hs = np.log(np.maximum(1 - eta, 1e-10))  # Tonks-like hard-core
    c1_p = np.log(np.maximum(rho_p, 1e-30)) + beta * phi_vals - mu_p + c1_hs
    c1_m = np.log(np.maximum(rho_m, 1e-30)) - beta * phi_vals - mu_m + c1_hs
    return c1_p, c1_m

def generate_mock_sample(Lz, n_bins_z, mu_p, mu_m, vext_p_fn, vext_m_fn, phi_fn, rng):
    """Generate a mock RPM density profile analytically (fast substitute for GCMC)."""
    z = np.linspace(0.5/n_bins_z*Lz, Lz*(1-0.5/n_bins_z), n_bins_z)
    beta_vext_p = np.array([vext_p_fn(zi) for zi in z])
    beta_vext_m = np.array([vext_m_fn(zi) for zi in z])
    phi_vals    = np.array([phi_fn(zi) for zi in z])
    
    # Wall exclusion: first/last 0.5σ = 0.5/Lz fraction of bins
    wall_bins = max(1, int(0.5 / (Lz / n_bins_z)))
    
    # Simple ideal-gas + packing correction
    rho_p_ideal = np.exp(mu_p - beta_vext_p - phi_vals)
    rho_m_ideal = np.exp(mu_m - beta_vext_m + phi_vals)
    
    # Soft normalisation to keep total packing < 0.7
    eta_ideal = rho_p_ideal + rho_m_ideal
    scale = np.where(eta_ideal > 0.7, 0.7 / eta_ideal, 1.0)
    rho_p = rho_p_ideal * scale
    rho_m = rho_m_ideal * scale
    
    # Apply wall exclusion
    rho_p[:wall_bins] = 0.0; rho_p[-wall_bins:] = 0.0
    rho_m[:wall_bins] = 0.0; rho_m[-wall_bins:] = 0.0
    
    # Add small noise
    rho_p = np.maximum(rho_p + rng.normal(0, 0.003, n_bins_z), 0)
    rho_m = np.maximum(rho_m + rng.normal(0, 0.003, n_bins_z), 0)
    
    c1_p, c1_m = make_mock_c1(rho_p, rho_m, mu_p, mu_m, phi_vals)
    
    return dict(z=z, rho_plus=rho_p, rho_minus=rho_m,
                c1_plus=c1_p, c1_minus=c1_m, phi=phi_vals)

print('Data generation functions defined.')

In [ ]:
import time

rng = np.random.default_rng(42)
Lx, Ly, Lz = 5.0, 5.0, 20.0
n_bins_z = 100

training_data = []
t0 = time.time()

for i in range(N_SAMPLES):
    mu_p = rng.uniform(-3.0, 0.0)
    mu_m = rng.uniform(-3.0, 0.0)
    vext_p_fn = generate_random_vext_ionic(Lz, rng)
    vext_m_fn = generate_random_vext_ionic(Lz, rng)
    phi_amplitude = rng.uniform(-0.5, 0.5)
    phi_fn = lambda z, A=phi_amplitude: A * np.sin(2*np.pi*z/Lz)
    
    if USE_MOCK_DATA:
        sample = generate_mock_sample(Lz, n_bins_z, mu_p, mu_m,
                                       vext_p_fn, vext_m_fn, phi_fn, rng)
    else:
        result = simulate_rpm(
            Lx=Lx, Ly=Ly, Lz=Lz,
            mu_plus=mu_p, mu_minus=mu_m,
            vext_plus_fn=vext_p_fn, vext_minus_fn=vext_m_fn,
            phi_fn=phi_fn,
            n_bins_z=n_bins_z, n_equil=500, n_prod=2000, rng=rng
        )
        beta_vext_p = np.array([vext_p_fn(z) for z in result['z']])
        beta_vext_m = np.array([vext_m_fn(z) for z in result['z']])
        phi_vals = result['phi']
        rho_p = result['rho_plus']
        rho_m = result['rho_minus']
        c1_p = (np.log(np.maximum(rho_p, 1e-30)) + beta_vext_p
                + phi_vals - mu_p)
        c1_m = (np.log(np.maximum(rho_m, 1e-30)) + beta_vext_m
                - phi_vals - mu_m)
        sample = dict(z=result['z'], rho_plus=rho_p, rho_minus=rho_m,
                      c1_plus=c1_p, c1_minus=c1_m, phi=phi_vals)
    
    training_data.append(sample)
    if (i+1) % 10 == 0:
        print(f'  Sample {i+1}/{N_SAMPLES}')

print(f'Generated {len(training_data)} samples in {time.time()-t0:.1f}s')

In [ ]:
# Plot a few representative density profiles
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, idx in zip(axes, [0, N_SAMPLES//2, N_SAMPLES-1]):
    s = training_data[idx]
    ax.plot(s['z'], s['rho_plus'],  'b-',  label=r'$\rho_+(z)$')
    ax.plot(s['z'], s['rho_minus'], 'r--', label=r'$\rho_-(z)$')
    ax.set_xlabel(r'$z/\sigma$')
    ax.set_ylabel(r'$\rho(z)$')
    ax.set_title(f'Sample {idx}')
    ax.legend(fontsize=8)
plt.suptitle('Representative density profiles from training set', y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Two-Channel Neural Functional

Following Bui & Cox, we train a **two-channel** neural network:
- **Input**: local windows of [ρ₊(z), ρ₋(z)] of width Δz = 3.6σ around z
- **Output**: c₁_R,ν(z) for species ν ∈ {+, -}
- **Architecture**: fully-connected MLP with Softplus activations

We train **two separate networks** — one for cations, one for anions — each taking both density channels as input (Bui & Cox: independent networks but same input features).

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} available')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not available — neural functional section will be skipped')

In [ ]:
if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn

    class IonicNeuralFunctional(nn.Module):
        """
        Two-channel neural functional for RPM.
        Input: concatenated [rho_plus_window, rho_minus_window]
        Output: c1 for one species (scalar per position)
        """
        def __init__(self, window_bins: int, hidden_dims=(64, 64, 64)):
            super().__init__()
            in_dim = 2 * window_bins  # two channels
            layers = []
            prev = in_dim
            for h in hidden_dims:
                layers += [nn.Linear(prev, h), nn.Softplus()]
                prev = h
            layers.append(nn.Linear(prev, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x).squeeze(-1)

    def make_windows(rho, window_bins):
        """Sliding windows with periodic padding. Returns (N, window_bins)."""
        N = len(rho)
        pad = window_bins // 2
        rho_padded = np.concatenate([rho[-pad:], rho, rho[:pad+1]])
        return np.lib.stride_tricks.sliding_window_view(rho_padded, window_bins)[:N]

    def build_dataset(data_list, window_width=3.6, dz=None):
        """Build (X, y) arrays for one species from training_data."""
        if dz is None:
            dz = float(data_list[0]['z'][1] - data_list[0]['z'][0])
        window_bins = 2 * int(round(window_width / (2 * dz))) + 1
        print(f'Window: {window_bins} bins = {window_bins*dz:.2f}σ')
        
        Xp_list, yp_list = [], []
        Xm_list, ym_list = [], []
        
        for s in data_list:
            rp, rm = s['rho_plus'], s['rho_minus']
            c1p, c1m = s['c1_plus'], s['c1_minus']
            
            wp = make_windows(rp, window_bins)
            wm = make_windows(rm, window_bins)
            X = np.concatenate([wp, wm], axis=1).astype(np.float32)  # (N, 2*wb)
            
            mask_p = np.isfinite(c1p) & (rp > 1e-4)
            mask_m = np.isfinite(c1m) & (rm > 1e-4)
            
            Xp_list.append(X[mask_p])
            yp_list.append(c1p[mask_p].astype(np.float32))
            Xm_list.append(X[mask_m])
            ym_list.append(c1m[mask_m].astype(np.float32))
        
        X_p = np.concatenate(Xp_list, axis=0)
        y_p = np.concatenate(yp_list, axis=0)
        X_m = np.concatenate(Xm_list, axis=0)
        y_m = np.concatenate(ym_list, axis=0)
        
        return window_bins, (X_p, y_p), (X_m, y_m)

    def train_model(X, y, window_bins, species_name, epochs=300, lr=1e-3, batch=256):
        """Train one IonicNeuralFunctional and return (model, train_losses)."""
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = IonicNeuralFunctional(window_bins).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=20)
        
        # Train/val split
        n = len(X)
        perm = np.random.permutation(n)
        n_val = max(1, int(0.1 * n))
        idx_val, idx_tr = perm[:n_val], perm[n_val:]
        
        X_t = torch.tensor(X[idx_tr], device=device)
        y_t = torch.tensor(y[idx_tr], device=device)
        X_v = torch.tensor(X[idx_val], device=device)
        y_v = torch.tensor(y[idx_val], device=device)
        
        train_losses, val_losses = [], []
        for epoch in range(epochs):
            model.train()
            perm_t = torch.randperm(len(X_t), device=device)
            epoch_loss = 0.0
            for start in range(0, len(X_t), batch):
                idx_b = perm_t[start:start+batch]
                loss = nn.functional.mse_loss(model(X_t[idx_b]), y_t[idx_b])
                opt.zero_grad(); loss.backward(); opt.step()
                epoch_loss += loss.item() * len(idx_b)
            epoch_loss /= len(X_t)
            train_losses.append(epoch_loss)
            
            model.eval()
            with torch.no_grad():
                vl = nn.functional.mse_loss(model(X_v), y_v).item()
            val_losses.append(vl)
            sched.step(vl)
            
            if (epoch+1) % 50 == 0:
                print(f'  [{species_name}] Epoch {epoch+1}/{epochs}: '
                      f'train={epoch_loss:.4f}, val={vl:.4f}')
        
        model.eval()
        return model, train_losses, val_losses

    print('IonicNeuralFunctional class defined.')

In [ ]:
if TORCH_AVAILABLE:
    dz_train = float(training_data[0]['z'][1] - training_data[0]['z'][0])
    window_bins, (X_p, y_p), (X_m, y_m) = build_dataset(
        training_data, window_width=3.6, dz=dz_train
    )
    print(f'Cation dataset: {X_p.shape[0]} points')
    print(f'Anion  dataset: {X_m.shape[0]} points')

In [ ]:
if TORCH_AVAILABLE:
    print('Training cation network...')
    model_plus,  losses_p_tr, losses_p_val = train_model(X_p, y_p, window_bins, 'cation',  epochs=200)
    print('Training anion network...')
    model_minus, losses_m_tr, losses_m_val = train_model(X_m, y_m, window_bins, 'anion', epochs=200)
    print('Training complete.')

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, (losses_tr, losses_val, name) in zip(axes, [
        (losses_p_tr, losses_p_val, 'Cation'),
        (losses_m_tr, losses_m_val, 'Anion'),
    ]):
        ep = np.arange(1, len(losses_tr)+1)
        ax.semilogy(ep, losses_tr,  label='Train')
        ax.semilogy(ep, losses_val, label='Validation', ls='--')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE loss')
        ax.set_title(f'{name} network')
        ax.legend()
    plt.tight_layout()
    plt.show()

## Part 5: Predictions and Validation

With trained networks, we can:
1. Predict c₁(z) from a density profile → compare to ground truth
2. Run DFT self-consistency to predict ρ(z) under new external conditions
3. Compute two-body correlations c₂(z,z') via autograd

In [ ]:
if TORCH_AVAILABLE:
    import torch

    def predict_c1_ionic(model, rho_p, rho_m, window_bins, device='cpu'):
        """Evaluate neural c1 for given (rho_plus, rho_minus) profiles."""
        wp = make_windows(rho_p, window_bins)
        wm = make_windows(rho_m, window_bins)
        X = np.concatenate([wp, wm], axis=1).astype(np.float32)
        X_t = torch.tensor(X, device=device)
        with torch.no_grad():
            c1 = model(X_t).cpu().numpy()
        return c1

    # Validate on a held-out sample
    s_test = training_data[-1]  # last sample as test
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model_plus.to(device); model_minus.to(device)

    c1p_pred = predict_c1_ionic(model_plus,  s_test['rho_plus'], s_test['rho_minus'],
                                  window_bins, device)
    c1m_pred = predict_c1_ionic(model_minus, s_test['rho_plus'], s_test['rho_minus'],
                                  window_bins, device)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (c1_true, c1_pred, label, color) in zip(axes, [
        (s_test['c1_plus'],  c1p_pred, 'Cation  c₁₊(z)', 'blue'),
        (s_test['c1_minus'], c1m_pred, 'Anion c₁₋(z)',   'red'),
    ]):
        mask = np.isfinite(c1_true)
        ax.plot(s_test['z'][mask], c1_true[mask], 'k-',  lw=1.5, label='Target')
        ax.plot(s_test['z'][mask], c1_pred[mask], color=color, ls='--', lw=1.5, label='Neural')
        ax.set_xlabel(r'$z/\sigma$')
        ax.set_ylabel(r'$c_1(z)$')
        ax.set_title(label)
        ax.legend()
    plt.suptitle('Neural functional predictions vs. target c₁ profiles', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
if TORCH_AVAILABLE:
    def dft_minimize_ionic(Lz, mu_p, mu_m, vext_p_fn, vext_m_fn, phi_fn,
                           model_plus, model_minus, window_bins,
                           n_bins=100, alpha=0.05, max_iter=5000, tol=1e-5,
                           z_wall=0.5, device='cpu'):
        """
        Picard iteration for two-species DFT self-consistency:
            ρ_ν(z) = exp(β·μ_ν - β·V_ν(z) - β·q_ν·ϕ(z) + c₁_ν(z; [ρ]))
        """
        dz = (Lz - 2*z_wall) / n_bins
        z  = np.linspace(z_wall + dz/2, Lz - z_wall - dz/2, n_bins)
        
        beta_vp  = np.array([vext_p_fn(zi) for zi in z])
        beta_vm  = np.array([vext_m_fn(zi) for zi in z])
        beta_phi = np.array([phi_fn(zi) for zi in z])
        
        # Initial guess: ideal gas
        rho_p = np.exp(mu_p - beta_vp - beta_phi)
        rho_m = np.exp(mu_m - beta_vm + beta_phi)
        rho_p = np.clip(rho_p, 0, 0.9)
        rho_m = np.clip(rho_m, 0, 0.9)
        
        for it in range(max_iter):
            c1p = predict_c1_ionic(model_plus,  rho_p, rho_m, window_bins, device)
            c1m = predict_c1_ionic(model_minus, rho_p, rho_m, window_bins, device)
            
            rho_p_new = np.exp(mu_p - beta_vp - beta_phi + c1p)
            rho_m_new = np.exp(mu_m - beta_vm + beta_phi + c1m)
            rho_p_new = np.clip(rho_p_new, 0, 0.95)
            rho_m_new = np.clip(rho_m_new, 0, 0.95)
            
            err = max(np.max(np.abs(rho_p_new - rho_p)),
                      np.max(np.abs(rho_m_new - rho_m)))
            rho_p = (1-alpha)*rho_p + alpha*rho_p_new
            rho_m = (1-alpha)*rho_m + alpha*rho_m_new
            
            if err < tol:
                print(f'DFT converged at iteration {it+1}')
                return z, rho_p, rho_m
        
        print(f'Warning: DFT did not converge (last err={err:.2e})')
        return z, rho_p, rho_m

    # Run DFT prediction for a new external condition
    test_mu_p, test_mu_m = -1.5, -1.5
    test_vext = lambda z: 0.3 * np.sin(4*np.pi*z/Lz)
    test_phi  = lambda z: 0.2 * np.cos(2*np.pi*z/Lz)

    z_dft, rho_p_dft, rho_m_dft = dft_minimize_ionic(
        Lz, test_mu_p, test_mu_m, test_vext, test_vext, test_phi,
        model_plus, model_minus, window_bins,
        n_bins=n_bins_z, alpha=0.03, device=device
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(z_dft, rho_p_dft, 'b-',  lw=2, label=r'$\rho_+(z)$ [neural DFT]')
    ax.plot(z_dft, rho_m_dft, 'r--', lw=2, label=r'$\rho_-(z)$ [neural DFT]')
    ax.set_xlabel(r'$z/\sigma$')
    ax.set_ylabel(r'$\rho(z)$ [$\sigma^{-3}$]')
    ax.set_title(f'Neural DFT prediction: RPM at βμ₊=βμ₋={test_mu_p}')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if TORCH_AVAILABLE:
    import torch

    def c2_ionic(model, rho_p, rho_m, window_bins, x_idx, device='cpu'):
        """
        Two-body correlation c₂(z_x, z') = dc₁(z_x) / dρ(z')
        via automatic differentiation.
        Returns arrays for dc1_plus/drho_plus, dc1_plus/drho_minus.
        """
        wp = make_windows(rho_p, window_bins)
        wm = make_windows(rho_m, window_bins)
        X_np = np.concatenate([wp, wm], axis=1).astype(np.float32)
        X = torch.tensor(X_np, device=device, requires_grad=False)

        # We want dc1(x_idx)/d rho(z') for all z'
        # Use jacobian at the single output x_idx wrt the density inputs
        rho_p_t = torch.tensor(rho_p.astype(np.float32), device=device, requires_grad=True)
        rho_m_t = torch.tensor(rho_m.astype(np.float32), device=device, requires_grad=True)

        # Recompute windows with grad tracking
        wp_t = torch.stack([torch.roll(rho_p_t, -i + window_bins//2)
                             for i in range(window_bins)], dim=1)
        wm_t = torch.stack([torch.roll(rho_m_t, -i + window_bins//2)
                             for i in range(window_bins)], dim=1)
        X_t = torch.cat([wp_t, wm_t], dim=1)

        c1 = model(X_t)
        c1[x_idx].backward()

        dc1_drho_p = rho_p_t.grad.cpu().numpy()
        dc1_drho_m = rho_m_t.grad.cpu().numpy()
        return dc1_drho_p, dc1_drho_m

    # Compute c2 at z_ref = middle of box
    z_ref_idx = n_bins_z // 2
    s = training_data[0]
    dc1p_drho_p, dc1p_drho_m = c2_ionic(
        model_plus, s['rho_plus'], s['rho_minus'], window_bins, z_ref_idx, device
    )

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(s['z'], dc1p_drho_p, 'b-', label=r'$\partial c^{(1)}_{+}/\partial \rho_+(z^\prime)$')
    axes[0].axvline(s['z'][z_ref_idx], color='k', ls=':', label=f'$z_{{ref}}$')
    axes[0].set_xlabel(r"$z'/\sigma$")
    axes[0].set_ylabel(r"$c^{(2)}_{++}(z_{ref}, z')$")
    axes[0].legend(fontsize=8)
    axes[0].set_title('Like-species c₂₊₊')

    axes[1].plot(s['z'], dc1p_drho_m, 'r-', label=r'$\partial c^{(1)}_{+}/\partial \rho_-(z^\prime)$')
    axes[1].axvline(s['z'][z_ref_idx], color='k', ls=':')
    axes[1].set_xlabel(r"$z'/\sigma$")
    axes[1].set_ylabel(r"$c^{(2)}_{+-}(z_{ref}, z')$")
    axes[1].legend(fontsize=8)
    axes[1].set_title('Unlike-species c₂₊₋')

    plt.suptitle(f'Two-body correlations from autograd at $z_{{ref}} = {s["z"][z_ref_idx]:.1f}\\sigma$',
                 y=1.02)
    plt.tight_layout()
    plt.show()

## Summary

We have implemented the Bui & Cox (2025) neural functional approach for ionic fluids:

| Step | Component | File |
|------|-----------|------|
| LMFT splitting | `v₀ = erfc(κr)/r` + `v₁ = erf(κr)/r` | `rpm_sim.py` |
| Mimic GCMC | `RPMSystem`, `simulate_rpm()` | `rpm_sim.py` |
| Mean-field correction | `lmft_potential()` | `rpm_sim.py` |
| Neural functional | `IonicNeuralFunctional` (2-channel MLP) | this notebook |
| DFT self-consistency | `dft_minimize_ionic()` | this notebook |
| c₂ via autograd | `c2_ionic()` | this notebook |

**Extensions** (see Bui & Cox paper):
- Size-asymmetric RPM (different σ₊ ≠ σ₋)
- Multivalent ions (|q| > 1)
- Bulk equation of state via radial projection of c₂
- Regularization with bulk two-body direct correlations

For the MLIP-MC integration, replace the ideal hard-sphere GCMC with `MLP_GCMC` for realistic pair potentials from a machine-learned force field.